In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q19/annotated/checkpoints/pre_cell_2.pickle

In [ ]:
%%RecordEvent
%%time
### cell 2 ###

# Define container and ship mode lists
SM_SMALL = [SMBOX, SMCASE, SMPACK, SMPKG]
MED_CONTAINER = [MEDBAG, MEDBOX, MEDPACK, MEDPKG]
LG_CONTAINER = [LGBOX, LGCASE, LGPACK, LGPKG]
SHIPMODES = [AIR, AIRREG]

# Filter lineitem with vectorized between and isin
li_qty = (
    lineitem.L_QUANTITY.between(4, 14)
    | lineitem.L_QUANTITY.between(15, 25)
    | lineitem.L_QUANTITY.between(26, 36)
)
li_ship = (lineitem.L_SHIPINSTRUCT == DELIVERINPERSON) & lineitem.L_SHIPMODE.isin(SHIPMODES)
flineitem = lineitem[li_qty & li_ship]

# Filter part using isin and between
small_cond = (
    (part.P_BRAND == Brand31)
    & part.P_CONTAINER.isin(SM_SMALL)
    & part.P_SIZE.between(1, 5)
)
med_cond = (
    (part.P_BRAND == Brand43)
    & part.P_CONTAINER.isin(MED_CONTAINER)
    & part.P_SIZE.between(1, 10)
)
lg_cond = (
    (part.P_BRAND == Brand43)
    & part.P_CONTAINER.isin(LG_CONTAINER)
    & part.P_SIZE.between(1, 15)
)
fpart = part[small_cond | med_cond | lg_cond]

# Join and apply final filtering
jn = flineitem.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")

cond1 = (
    (jn.P_BRAND == Brand31)
    & jn.P_CONTAINER.isin(SM_SMALL)
    & jn.L_QUANTITY.between(4, 14)
    & jn.P_SIZE.le(5)
)
cond2 = (
    (jn.P_BRAND == Brand43)
    & jn.P_CONTAINER.isin(MED_CONTAINER)
    & jn.L_QUANTITY.between(15, 25)
    & jn.P_SIZE.le(10)
)
cond3 = (
    (jn.P_BRAND == Brand43)
    & jn.P_CONTAINER.isin(LG_CONTAINER)
    & jn.L_QUANTITY.between(26, 36)
    & jn.P_SIZE.le(15)
)
jn = jn[cond1 | cond2 | cond3]

# Compute total
total = (jn.L_EXTENDEDPRICE * (1.0 - jn.L_DISCOUNT)).sum()

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q19/rewritten/o4_mini_high/checkpoints/post_cell_2_try_0.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/tpch/notebooks/q19/opt_cell_exec_info_2_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [ ]:
opt_output = Out.get(4)